# Day 7 — Transfer Learning & Object Detection | RetailMind AI

This notebook upgrades the Day 6 vision module with:
1. ResNet50 transfer learning for fashion product classification.
2. Fine-tuning the top pretrained layers.
3. Ultralytics YOLO inference for object detection and bounding boxes.
4. Evaluation, CSV exports, and saved model artifacts.

**Dataset:** Fashion Product Images Small from Kaggle. Place the extracted files here:

```text
data/fashion_product_images/
├── images/
└── styles.csv
```

The classifier predicts `masterCategory`. The pretrained YOLO COCO model detects general COCO objects; it is not a fine-grained fashion detector.


## 1. Install and import libraries

In [ ]:
# %pip install -q tensorflow ultralytics

In [1]:
import csv
import json
import random
import shutil
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

from ultralytics import YOLO

print("TensorFlow:", tf.__version__)

I0000 00:00:1794236230.419961   12464 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794236230.690910   12464 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794236232.564517   12464 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


ModuleNotFoundError: No module named 'ultralytics'

## 2. Project paths and reproducibility

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CWD = Path.cwd()
ROOT_DIR = CWD.parent if (CWD.parent / "data").exists() else CWD
DATA_DIR = ROOT_DIR / "data"
PRODUCT_DIR = DATA_DIR / "fashion_product_images"
IMAGE_DIR = PRODUCT_DIR / "images"
METADATA_PATH = PRODUCT_DIR / "styles.csv"

MODEL_DIR = ROOT_DIR / "models" / "vision"
DETECTION_DIR = ROOT_DIR / "outputs" / "day7_detection"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DETECTION_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT_DIR)
print("Model directory:", MODEL_DIR)

## 3. Load metadata and retain images that exist

In [ ]:
if not IMAGE_DIR.exists() or not METADATA_PATH.exists():
    raise FileNotFoundError(
        "Expected data/fashion_product_images/images/ and styles.csv. "
        "Download and extract the Kaggle dataset first."
    )

records = []
with open(METADATA_PATH, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    required = {"id", "masterCategory"}
    if not reader.fieldnames or not required.issubset(reader.fieldnames):
        raise ValueError("styles.csv must contain id and masterCategory.")

    for row in reader:
        image_id = (row.get("id") or "").strip().removesuffix(".0")
        category = (row.get("masterCategory") or "").strip()
        if image_id and category:
            image_path = IMAGE_DIR / f"{image_id}.jpg"
            if image_path.is_file():
                records.append((str(image_path), category))

if not records:
    raise ValueError("No usable product images found.")

CLASS_NAMES = sorted({label for _, label in records})
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

print("Usable images:", len(records))
print("Categories:", CLASS_NAMES)

## 4. Create stratified train, validation, and test splits

In [ ]:
rng = random.Random(SEED)
train_records, val_records, test_records = [], [], []

for category in CLASS_NAMES:
    category_records = [r for r in records if r[1] == category]
    rng.shuffle(category_records)

    n = len(category_records)
    train_end = int(n * 0.8)
    val_end = int(n * 0.9)

    train_records.extend(category_records[:train_end])
    val_records.extend(category_records[train_end:val_end])
    test_records.extend(category_records[val_end:])

rng.shuffle(train_records)
rng.shuffle(val_records)
rng.shuffle(test_records)

def encode_records(items):
    paths = [item[0] for item in items]
    labels = np.asarray([CLASS_TO_INDEX[item[1]] for item in items], dtype=np.int32)
    return paths, labels

train_paths, train_labels = encode_records(train_records)
val_paths, val_labels = encode_records(val_records)
test_paths, test_labels = encode_records(test_records)

print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))

## 5. Build the ResNet50 input pipeline

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label, training=False):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)

    if training:
        image = tf.image.random_flip_left_right(image)
        image = tf.image.random_brightness(image, max_delta=15.0)
        image = tf.image.random_contrast(image, lower=0.9, upper=1.1)

    image = preprocess_input(image)
    return image, label

def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(min(len(paths), 10000), seed=SEED,
                        reshuffle_each_iteration=True)
    ds = ds.map(
        lambda path, label: load_image(path, label, training),
        num_parallel_calls=AUTOTUNE
    )
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels)
test_ds = make_dataset(test_paths, test_labels)
print("Datasets ready.")

## 6. Build the transfer-learning model

In [ ]:
def build_transfer_model(input_shape, num_classes):
    backbone = ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape
    )
    backbone.trainable = False

    inputs = keras.Input(shape=input_shape)
    features = backbone(inputs, training=False)
    features = layers.GlobalAveragePooling2D()(features)
    features = layers.Dropout(0.3)(features)
    outputs = layers.Dense(num_classes, activation="softmax")(features)

    model = keras.Model(inputs, outputs, name="retailmind_resnet50")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model, backbone

transfer_model, backbone = build_transfer_model(
    (224, 224, 3), NUM_CLASSES
)
transfer_model.summary()

## 7. Train the classification head with frozen ResNet50

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

head_history = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping],
    verbose=1
)

## 8. Fine-tune the top ResNet50 layers

In [ ]:
# Unfreeze the backbone, then freeze all but its last 30 layers.
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False

# Keep BatchNormalization layers frozen for stable fine-tuning.
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Recompile after changing trainable layers. Use a small learning rate.
transfer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

fine_tune_history = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
            verbose=1
        )
    ],
    verbose=1
)

## 9. Evaluate the fine-tuned classifier

In [ ]:
test_loss, test_accuracy = transfer_model.evaluate(test_ds, verbose=1)
print(f"ResNet50 test loss: {test_loss:.4f}")
print(f"ResNet50 test accuracy: {test_accuracy:.4f}")

probabilities = transfer_model.predict(test_ds, verbose=1)
predictions = np.argmax(probabilities, axis=1)

## 10. Save confusion matrix and per-class accuracy as CSV

In [ ]:
confusion = tf.math.confusion_matrix(
    test_labels, predictions, num_classes=NUM_CLASSES
).numpy()

confusion_path = MODEL_DIR / "resnet50_confusion_matrix.csv"
with open(confusion_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Actual / Predicted"] + CLASS_NAMES)
    for name, row in zip(CLASS_NAMES, confusion):
        writer.writerow([name] + row.tolist())

class_accuracy = np.diag(confusion) / np.maximum(confusion.sum(axis=1), 1)
with open(MODEL_DIR / "resnet50_class_accuracy.csv", "w",
          newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["Category", "Accuracy"])
    writer.writerows(zip(CLASS_NAMES, class_accuracy))

with open(MODEL_DIR / "resnet50_metrics.csv", "w",
          newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["Model", "Test Loss", "Test Accuracy"])
    writer.writeheader()
    writer.writerow({
        "Model": "ResNet50 fine-tuned",
        "Test Loss": float(test_loss),
        "Test Accuracy": float(test_accuracy)
    })

print("Saved classifier evaluation CSV files.")

## 11. Compare with the Day 6 CNN

In [ ]:
day6_metrics_path = MODEL_DIR / "fashion_product_metrics.csv"
comparison_rows = []

if day6_metrics_path.exists():
    with open(day6_metrics_path, "r", encoding="utf-8", newline="") as file:
        day6_rows = list(csv.DictReader(file))
    if day6_rows and "Test Accuracy" in day6_rows[0]:
        comparison_rows.append({
            "Model": "Day 6 CNN",
            "Test Accuracy": day6_rows[0]["Test Accuracy"]
        })
else:
    print("Day 6 metrics CSV not found. Run the Day 6 product CNN evaluation first.")

comparison_rows.append({
    "Model": "Day 7 ResNet50 fine-tuned",
    "Test Accuracy": float(test_accuracy)
})

comparison_path = MODEL_DIR / "day7_classifier_comparison.csv"
with open(comparison_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["Model", "Test Accuracy"])
    writer.writeheader()
    writer.writerows(comparison_rows)

print(comparison_rows)
print("Comparison saved:", comparison_path)

## 12. Save classifier and category mapping

In [ ]:
classifier_path = MODEL_DIR / "fashion_resnet50_finetuned.keras"
transfer_model.save(classifier_path)

class_mapping_path = MODEL_DIR / "fashion_product_classes.json"
with open(class_mapping_path, "w", encoding="utf-8") as file:
    json.dump(CLASS_NAMES, file, indent=4)

print("Saved classifier:", classifier_path)
print("Saved classes:", class_mapping_path)

## 13. Load pretrained YOLO detector

The `yolo11n.pt` checkpoint is pretrained on COCO. It can detect general classes such as `handbag`, `backpack`, `suitcase`, and `person`, depending on the image. It is not trained to classify all apparel types.


In [ ]:
detector = YOLO("yolo11n.pt")
print("YOLO loaded.")
print("Example COCO class names:", list(detector.names.values())[:20])

## 14. Run YOLO on a catalog image

In [ ]:
if not test_paths:
    raise ValueError("No test images available for YOLO inference.")

detection_image = test_paths[0]
results = detector.predict(
    source=detection_image,
    conf=0.25,
    save=True,
    project=str(DETECTION_DIR),
    name="single_image",
    exist_ok=True,
    verbose=False
)

result = results[0]
print("Image:", detection_image)
print("Annotated image directory:", result.save_dir)

## 15. Extract bounding boxes and export detections

In [ ]:
detection_rows = []

if result.boxes is not None:
    boxes = result.boxes.xyxy.cpu().numpy()
    scores = result.boxes.conf.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy().astype(int)

    for box, score, class_id in zip(boxes, scores, class_ids):
        x1, y1, x2, y2 = box.tolist()
        detection_rows.append({
            "image": Path(detection_image).name,
            "class_id": int(class_id),
            "class_name": detector.names[int(class_id)],
            "confidence": float(score),
            "x1": float(x1), "y1": float(y1),
            "x2": float(x2), "y2": float(y2)
        })

detection_csv = DETECTION_DIR / "catalog_detections.csv"
fields = ["image", "class_id", "class_name", "confidence",
          "x1", "y1", "x2", "y2"]

with open(detection_csv, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fields)
    writer.writeheader()
    writer.writerows(detection_rows)

print("Detected objects:", len(detection_rows))
print("CSV saved:", detection_csv)
for row in detection_rows:
    print(row)

## 16. Run YOLO on a small batch of catalog images

In [ ]:
NUM_DETECTION_IMAGES = min(20, len(test_paths))
batch_rows = []

for image_path in test_paths[:NUM_DETECTION_IMAGES]:
    batch_results = detector.predict(
        source=image_path,
        conf=0.25,
        save=True,
        project=str(DETECTION_DIR),
        name="batch",
        exist_ok=True,
        verbose=False
    )
    batch_result = batch_results[0]

    if batch_result.boxes is None:
        continue

    boxes = batch_result.boxes.xyxy.cpu().numpy()
    scores = batch_result.boxes.conf.cpu().numpy()
    class_ids = batch_result.boxes.cls.cpu().numpy().astype(int)

    for box, score, class_id in zip(boxes, scores, class_ids):
        x1, y1, x2, y2 = box.tolist()
        batch_rows.append({
            "image": Path(image_path).name,
            "class_name": detector.names[int(class_id)],
            "confidence": float(score),
            "x1": float(x1), "y1": float(y1),
            "x2": float(x2), "y2": float(y2)
        })

batch_csv = DETECTION_DIR / "batch_detections.csv"
batch_fields = ["image", "class_name", "confidence", "x1", "y1", "x2", "y2"]
with open(batch_csv, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=batch_fields)
    writer.writeheader()
    writer.writerows(batch_rows)

print("Images processed:", NUM_DETECTION_IMAGES)
print("Total detections:", len(batch_rows))
print("Saved:", batch_csv)

## 17. Save detector checkpoint and vision-module configuration

In [ ]:
# Ultralytics downloads weights to its cache. If a local checkpoint exists,
# copy it into the project so the artifact is self-contained.
local_checkpoint = Path("yolo11n.pt")
saved_checkpoint = MODEL_DIR / "yolo11n.pt"

if local_checkpoint.exists():
    shutil.copy2(local_checkpoint, saved_checkpoint)
    detector_weights_name = saved_checkpoint.name
else:
    detector_weights_name = "yolo11n.pt"
    print("YOLO weights remain in the Ultralytics cache; reload with YOLO('yolo11n.pt').")

vision_config = {
    "classifier": {
        "architecture": "ResNet50",
        "pretrained_weights": "ImageNet",
        "input_shape": [224, 224, 3],
        "classes": CLASS_NAMES,
        "model_file": classifier_path.name
    },
    "detector": {
        "architecture": "YOLO11n",
        "pretrained_dataset": "COCO",
        "weights": detector_weights_name,
        "note": "General COCO detector, not a fine-grained fashion detector."
    }
}

config_path = MODEL_DIR / "day7_vision_config.json"
with open(config_path, "w", encoding="utf-8") as file:
    json.dump(vision_config, file, indent=4)

print("Vision configuration saved:", config_path)

## 18. Final Day 7 summary

In [ ]:
print(f"ResNet50 test accuracy: {test_accuracy:.4f}")
print(f"ResNet50 test loss: {test_loss:.4f}")
print(f"YOLO images processed: {NUM_DETECTION_IMAGES}")
print(f"YOLO detections: {len(batch_rows)}")

print("\nModel artifacts:")
for path in sorted(MODEL_DIR.iterdir()):
    if path.is_file():
        print("-", path.name)

print("\nDetection outputs:")
for path in sorted(DETECTION_DIR.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(ROOT_DIR))

## 19. README update

After running the notebook, record the actual measured accuracy values in your main `README.md`.

```markdown
## Day 7 — Transfer Learning & Object Detection

### Transfer Learning
- Backbone: ResNet50 pretrained on ImageNet
- Strategy: frozen feature extractor, then fine-tune top 30 layers
- Dataset: Fashion Product Images Small
- Target: masterCategory

### Object Detection
- Framework: Ultralytics YOLO
- Model: YOLO11n pretrained on COCO
- Output: bounding boxes, class names, confidence scores
- Limitation: COCO labels are general object classes, not detailed fashion categories

### Results
| Model | Test Accuracy |
|---|---:|
| Day 6 CNN | ADD_ACTUAL_RESULT |
| Day 7 ResNet50 | ADD_ACTUAL_RESULT |

### Artifacts
- models/vision/fashion_resnet50_finetuned.keras
- models/vision/fashion_product_classes.json
- models/vision/resnet50_confusion_matrix.csv
- models/vision/resnet50_metrics.csv
- models/vision/day7_classifier_comparison.csv
- models/vision/day7_vision_config.json
- outputs/day7_detection/
```

### Notes
- For a strict Day 6 vs Day 7 comparison, both models must use the exact same train/validation/test split. The Day 7 split here is stratified; if Day 6 used a different split, compare the results as indicative only.
- This notebook demonstrates YOLO inference, not detection-model evaluation. Calculating mAP requires ground-truth bounding-box annotations.
- To detect fashion-specific items reliably, collect annotated bounding boxes and fine-tune YOLO on those classes.
- If memory is limited, reduce `BATCH_SIZE` to 16.
